 # 1. Imports & Setup

 In this cell, we import all required modules and connect to ComfyUI.

In [ ]:
import asyncio
import json
import random
from pathlib import Path
from IPython.display import clear_output, display
# ComfyScript imports
import comfy_script.ui as ui
from comfy_script.runtime import Workflow
from comfy_script.runtime import *
from comfy_script.runtime.nodes import *

# Connect to ComfyUI
load("http://127.0.0.1:8188/")
#load("/workspace/ComfyUI/")

# Import our custom modules
from jupyter_data_types import GroupsConfig, GenItem
from jupyter_utils import work_with, save_img
from jupyter_workflows import flux1_img_gen


 # 2. Load Workload (JSON)

 Load JSON with groups and prompts. The data is parsed via Pydantic.

In [ ]:
WORKLOAD_FILE_PATH = Path("../flux_workload.json")
with open(WORKLOAD_FILE_PATH, "r", encoding="utf-8") as f:
    JSON_PAYLOAD = f.read()

# Parsing and saving as a global variable
WORKLOAD_CONFIG = GroupsConfig.model_validate_json(JSON_PAYLOAD)
print(f"Loaded {len(WORKLOAD_CONFIG.groups)} groups.")
prompts = 0
for g in WORKLOAD_CONFIG.groups:
    prompts+=len(g.prompts)

print(f"Loaded {prompts} prompts.")

 # 3. Global Generation Parameters

 Global variables that will be passed to flux_img_gen.

In [ ]:
get_seed = lambda: random.randint(0, 1000000)
SEED = -1 #-1 for generate seed
FLUX_MODEL = "flux1-schnell.safetensors"
FLUX_VAE = "flux_v1_vae.safetensors"
CLIP1 = "clip_l.safetensors"
#CLIP1 = "t5xxl_fp8_e4m3fn.safetensors"
CLIP2 = "t5xxl_fp16.safetensors"
# CLIP1 = "t5xxl_fp16.safetensors"
GUIDANCE = 1
WEIGHT_DTYPE = "fp8_e4m3fn"
KSAMPLER = "euler"
MAX_SHIFT = 1.15
BASE_SHIFT = 0.5
SCHEDULER = "simple"
STEPS = 4
DENOISE = 1.0
LATENT_WIDTH = 1024
LATENT_HEIGHT = 1024
BATCH_SIZE = 1

OUTPUT_DIR = Path("/workspace/export/flux-prompts-fp16/")


 # 4. Execution & Preview

 Using work_with to traverse JSON, generate, save, and dynamically update the preview.

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display
import random

# --- Ініціалізація глобальних лічильників ---
current_group = None
group_images = []
group_titles = []
total_generated_globally = 0

total_groups = len(WORKLOAD_CONFIG.groups)
total_prompts = sum(len(g.prompts) for g in WORKLOAD_CONFIG.groups)

# --- Створення ізольованих віджетів виводу ---
grid_output = widgets.Output()
status_output = widgets.Output()

# Рендеримо зони в інтерфейсі (сітка зверху, статус знизу)
display(grid_output)
display(widgets.HTML("<hr style='border: 1px solid #30363d; margin: 15px 0;'>"))
display(status_output)


def render_image_grid(images, titles):
    """Генерує чисту адаптивну сітку картинок за допомогою native ipywidgets"""
    grid_items = []
    for img, title in zip(images, titles):
        # Конвертуємо PIL Image в байти для віджета
        byte_arr = io.BytesIO()
        img.save(byte_arr, format='PNG')
        img_bytes = byte_arr.getvalue()
        
        img_widget = widgets.Image(
            value=img_bytes, 
            format='png', 
            layout=widgets.Layout(width='100%', max_width='250px', border='1px solid #30363d', border_radius='4px')
        )
        lbl_widget = widgets.Label(
            value=title, 
            layout=widgets.Layout(display='flex', justify_content='center', font_weight='bold')
        )
        
        grid_items.append(widgets.VBox([img_widget, lbl_widget], layout=widgets.Layout(align_items='center')))
        
    return widgets.GridBox(
        grid_items, 
        layout=widgets.Layout(
            grid_template_columns='repeat(auto-fill, minmax(220px, 1fr))',
            grid_gap='15px',
            width='100%'
        )
    )


async def process_item(item: GenItem):
    global current_group, group_images, group_titles, total_generated_globally

    # Кількість промптів у поточній групі
    prompts_in_group = len(WORKLOAD_CONFIG.groups[item.group_index].prompts)

    # 1. Якщо змінилася група — скидаємо її локальну сітку
    if current_group != item.group_name:
        current_group = item.group_name
        group_images = []
        group_titles = []
        with grid_output:
            grid_output.clear_output(wait=True)

    # 2. Оновлюємо СТАТУС (до початку генерації, щоб бачити, що саме зараз печеться)
    with status_output:
        status_output.clear_output(wait=True)
        print("📊 СТАН ГЕНЕРАЦІЇ:")
        print(f"Загальний прогрес: груп {item.group_index}/{total_groups} | картинок {total_generated_globally}/{total_prompts}")
        print(f"Зараз генерується: Group {item.group_index + 1}/{total_groups} image {item.prompt_index + 1}/{prompts_in_group}")
        print(f"Назва таски: {item.item_name}")
        print(f"Промпт: {item.prompt.positive}")

    # 3. Сама генерація
    img = await flux1_img_gen(
        prompt=item.prompt.positive,
        flux_model=FLUX_MODEL,
        flux_vae=FLUX_VAE,
        clip1=CLIP1,
        clip2=CLIP2,
        noise=SEED if SEED != -1 else get_seed(),
        guidance=GUIDANCE,
        weight_dtype=WEIGHT_DTYPE,
        ksampler=KSAMPLER,
        max_shift=MAX_SHIFT,
        base_shift=BASE_SHIFT,
        scheduler=SCHEDULER,
        steps=STEPS,
        denoise=DENOISE,
        latent_width=LATENT_WIDTH,
        latent_height=LATENT_HEIGHT,
        batch_size=BATCH_SIZE,
    )

    if img:
        # Збереження на диск
        save_path = OUTPUT_DIR / item.group_name / f"{item.item_name}.png"
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_img(img, save_path)

        # Оновлення метрик
        total_generated_globally += 1
        group_images.append(img)
        group_titles.append(item.item_name)

        # 4. Оновлюємо СІТКУ поточної групи (без зачіпання зони статусу)
        with grid_output:
            grid_output.clear_output(wait=True)
            display(render_image_grid(group_images, group_titles))

        # 5. Фінальний апдейт статусу після успішного завершення цієї картинки
        with status_output:
            status_output.clear_output(wait=True)
            print("📊 СТАН ГЕНЕРАЦІЇ:")
            print(f"Загальний прогрес: груп {item.group_index + 1}/{total_groups} | картинок {total_generated_globally}/{total_prompts}")
            print(f"Остання готова: Group {item.group_index + 1}/{total_groups} image {item.prompt_index + 1}/{prompts_in_group}")


# --- Запуск пулу задач ---
callbacks = work_with(WORKLOAD_CONFIG, process_item)
for cb in callbacks:
    await cb()